# 48连续季度原始SEC申报文本核对
两套旧元数据字段并行归一化；当前、上季及去年同季GAAP diluted EPS均与原发行人语料核对，不补缺或改变旧数值。

In [1]:
from pathlib import Path
import os,json,hashlib
import pandas as pd,qlib
from qlib.workflow import R
from src.workflow.minute_action_value import _source
from src.data.sec_eps_text import extract_sec_eps_text,quarterly_filing_identity
from src.data.nas_market_data import archive_nas_recorder,materialize_nas_evidence
root=Path('/Users/rziiiii/dev/workspace');card=root/'research/nvda_sec_eps_series_20261015';p=json.loads((card/'plan.json').read_text());provider=root/'data/qlib/nvda_eps_corpus_20261015_v1';os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(provider),region='us');source,source_audit=_source(root,p['corpus_source'],['analysis/events']);events={e['fiscal_quarter']:e for e in source['analysis/events']}
if (root/p['raw_dir']/'metadata.json').exists():
 raw={e['file']:(root/p['raw_dir']/e['file']).read_bytes() for e in p['cases']};metadata_bytes=(root/p['raw_dir']/'metadata.json').read_bytes()
else:
 old=json.loads((card/'summary.json').read_text())
 if not Path(old['recorder_local_dir']).exists():materialize_nas_evidence(card/'nas_binding.json',card=card)
 with R.uri_context(str(card/'qlib_recorder')):
  prior=R.get_recorder(recorder_id=old['recorder_id'],experiment_id=old['experiment_id']);raw=prior.load_object('inputs/web_text_bytes');metadata_bytes=prior.load_object('inputs/metadata_bytes')
assert hashlib.sha256(metadata_bytes).hexdigest()==p['metadata_sha256'];metadata=json.loads(metadata_bytes);rows=[]
for case in p['cases']:
 payload=raw[case['file']];assert hashlib.sha256(payload).hexdigest()==case['sha256'];identity=quarterly_filing_identity(metadata[case['quarter']]);assert identity['accession']==case['accession'];parsed=extract_sec_eps_text(payload.decode(),quarter=case['quarter'],accession=case['accession']);original=events[case['quarter']]
 for field in ['current_eps','prior_quarter_comparative_eps','prior_year_comparative_eps']:assert parsed[field]==original[field],(case['quarter'],field,parsed[field],original[field])
 rows.append({**parsed,'release_date':original['release_date'],'filing_date':identity['filing_date'],'api_accepted_raw':identity['api_accepted_raw'],'index_accepted_raw':identity['index_accepted_raw'],'url_discovery':case['url_discovery'],'text_sha256':case['sha256'],'original_html_bytes_retained':False,'first_publication_clock_verified':False,'training_input_qualified':False})
table=pd.DataFrame(rows).sort_values('quarter');sequence=table.quarter.map(lambda q:4*int(q[2:6])+int(q[-1]));assert len(table)==48 and sequence.diff().dropna().eq(1).all();assert table.release_date.eq(pd.Series([events[q]['release_date'] for q in table.quarter],index=table.index)).all();policy={'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/verified_facts']}
with R.start(experiment_name=card.name,uri=str(card/'qlib_recorder')):
 rec=R.get_recorder();rec.save_objects(artifact_path='inputs',plan=p,web_text_bytes=raw,metadata_bytes=metadata_bytes,source_audit=source_audit,parser=(root/'src/data/sec_eps_text.py').read_text());rec.save_objects(artifact_path='analysis',verified_facts=table);rec.log_metrics(continuous_quarters=len(table),new_fits=0,new_backtests=0)
summary={'experiment':card.name,'decision':'continuous_original_accession_text_checked_needs_clock_qualification','new_fits':0,'new_backtests':0,'continuous_quarters':len(table),'first_quarter':table.quarter.iloc[0],'last_quarter':table.quarter.iloc[-1],'first_release_date':table.release_date.min(),'last_release_date':table.release_date.max(),'all_three_eps_fields_match_issuer_corpus':True,'original_html_bytes_retained':False,'training_input_qualified':False,'provider_uri':str(provider),'region':'us','as_of':'2026-10-14','recorder_id':rec.id,'experiment_id':str(rec.experiment_id),'recorder_uri':str(card/'qlib_recorder'),'recorder_local_dir':str(card/'qlib_recorder'/str(rec.experiment_id)/rec.id),'artifact_policy':policy,'objective_achieved':False,'scope':p['scope']};rec.save_objects(artifact_path='analysis',summary=summary);a=Path(summary['recorder_local_dir'])/'artifacts';rec.log_metrics(registered_artifact_count=sum(f.is_file() for f in a.rglob('*')));table.to_csv(card/'verified_quarters.csv',index=False)
import yaml
(card/'workflow.yaml').write_text(yaml.safe_dump({'experiment':card.name,'provider_uri':str(provider),'artifact_policy':policy}));(card/'verification_manifest.json').write_text(json.dumps({'recorder_id':rec.id,'entries':[{'path':str(f.relative_to(card)),'sha256':hashlib.sha256(f.read_bytes()).hexdigest()} for f in sorted(a.rglob('*')) if f.is_file()]},indent=2));summary['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=str(rec.experiment_id));(card/'summary.json').write_text(json.dumps(summary,indent=2));print(summary)


# 原始SEC申报EPS：48连续季度文本已核对

本轮补齐FY2015Q3–FY2027Q2共48个连续季度，公告日期2014-11-06至2026-08-26。每季当期、上季比较、上年同季比较三个GAAP diluted EPS共144字段，逐值与原发行人公告语料一致。Q4季度值与全年值、GAAP与Non-GAAP明确分开，不用年度EPS减九个月EPS凑季度。

此前可见来源数量被低估：旧记录有reported_fiscal_period/observed_exhibit_links和reported_fiscal_quarter/observed_exhibit_url两套字段。现归一化季度及申报编号、排除预告更新记录，保留原数据。已观察URL与按既有申报编号推定文件名后验证正文的URL分别标识；未声称全部索引都已读取。FY2017Q2/Q3旧文件名候选失败，实际q2fy17pr.htm/q3fy17pr.htm可读，正文季度和数值均已核对，不能把推定成功扩成对其他未访问URL的事实。

源码sec_eps_text.py只读取保留的工具文本，核验SEC原始申报编号URL、GAAP表及三个季度栏位；不从Non-GAAP、全年、新闻引用或下一年比较值猜原值。FY2017Q1=0.33、Q2=0.40的原报告值继续保留；后来比较栏中的0.35/0.41仍是另外的当期可比字段。前轮确认的会计调整未向过去回填。2021/2024拆股口径仍沿用独立账本，文本数值未统一后复权。

所有48份实际web工具响应、合并的原元数据、解析器和144字段结果已进Qlib Recorder/NAS；根目录仅计划、索引、小表与结论。这些是原申报URL当前返回的渲染文本，不是原始HTML字节或当年缓存镜像。保留filed、release、API accepted、index accepted原字符串，不自行推断其时区等价；完整时点资格仍未签发。

决定continuous_original_accession_text_checked_needs_clock_qualification，training_input_qualified=false。数值连续性问题已解决，不能继续称“只有两个不连续样本”；尚需固定可用日期政策、核对原始申报与首次披露关系，才能构造不泄漏的季节性特征。需要精确公告反应时点的模型与只在保守延迟后使用数据的模型应分开验证，不把延迟假设叫作首收时间认证。

本轮0拟合、0回测、未计算SUE或查看事件后收益，生产和原收益未改，稳定净SOXX超额未完成。下一实质步骤是时间绑定与固定特征合同，随后才进行配对训练；不重开此前一致预期/营收指引失败路线，不用本轮来源扩充宣称alpha。

notebook已执行48季度连续性及144字段一致性核对；1项新解析合同测试通过（两套元数据、季度/全年、GAAP/Non-GAAP及非初次更新排除）。NAS冷恢复逐对象哈希通过。全部原始链接及字段见verified_quarters.csv，已知时钟缺口保留为false。
